# Correlations

In [4]:
import numpy as np
from scipy.stats import kendalltau, spearmanr
import pandas as pd
from google.colab import files

In [6]:
print("Upload bleu_scores.txt, comet_scores.txt, our_metric_scores.txt, human_scores.txt")
uploaded = files.upload()

# Extract filenames (Colab stores them in uploaded.keys())
bleu_file = "bleu_scores.txt"
comet_file = "comet_scores.txt"
our_metric_file = "our_metric_scores.txt"
human_file = "human_scores.txt"

Upload bleu_scores.txt, comet_scores.txt, our_metric_scores.txt, human_scores.txt


Saving human_scores.txt to human_scores.txt
Saving our_metric_scores.txt to our_metric_scores.txt
Saving comet_scores.txt to comet_scores.txt
Saving bleu_scores.txt to bleu_scores.txt


In [9]:
class Correlation:
    def __init__(self, bleu_file, comet_file, our_metric_file, human_file):
        self.bleu_scores = self._load_scores(bleu_file)
        self.comet_scores = self._load_scores(comet_file)
        self.our_scores = self._load_scores(our_metric_file)
        self.human_scores = self._load_scores(human_file)

    def _load_scores(self, file_path):
        """Load scores assuming one score per line."""
        with open(file_path, "r", encoding="utf-8") as f:
            return np.array([float(line.strip()) for line in f if line.strip()])

    def compute_correlation(self, metric_scores):
        """Compute Kendall τ and Spearman ρ correlations."""
        kendall, _ = kendalltau(metric_scores, self.human_scores)
        spearman, _ = spearmanr(metric_scores, self.human_scores)
        avg = (kendall + spearman) / 2
        return kendall, spearman, avg

    def save_results(self, output_file="correlation_results.tsv"):
        """Compute all correlations and save to TSV."""
        results = []

        metrics = {
            "BLEU": self.bleu_scores,
            "COMET": self.comet_scores,
            "OUR_METRIC": self.our_scores
        }

        for name, scores in metrics.items():
            kendall, spearman, avg = self.compute_correlation(scores)
            results.append({
                "metric": name,
                "kendall_tau": round(kendall,2),
                "spearman_rho": round(spearman,2),
                "average": round(avg,2)
            })

        df = pd.DataFrame(results)
        df.to_csv(output_file, sep="\t", index=False)
        print(f"Saved correlation results to {output_file}")

        files.download(output_file)

In [10]:
cor = Correlation(
    bleu_file=bleu_file,
    comet_file=comet_file,
    our_metric_file=our_metric_file,
    human_file=human_file
)

cor.save_results("correlation_results.tsv")

Saved correlation results to correlation_results.tsv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Similarity

In [11]:
class Similarity:
    def __init__(self, bleu_file, comet_file, our_metric_file, human_file):
        self.bleu_scores = self._load_scores(bleu_file)
        self.comet_scores = self._load_scores(comet_file)
        self.our_scores = self._load_scores(our_metric_file)
        self.human_scores = self._load_scores(human_file)

    def _load_scores(self, file_path):
        """Load scores assuming one score per line."""
        with open(file_path, "r", encoding="utf-8") as f:
            return np.array([float(line.strip()) for line in f if line.strip()])

    def compute_similarity(self, metric_scores):
        """Compute MAE and MSE against human scores."""
        mae = np.mean(np.abs(metric_scores - self.human_scores))
        mse = np.mean((metric_scores - self.human_scores) ** 2)
        avg = (mae + mse) / 2
        return mae, mse, avg

    def save_results(self, output_file="similarity_results.tsv"):
        """Compute MAE/MSE and save to TSV."""
        results = []

        metrics = {
            "BLEU": self.bleu_scores,
            "COMET": self.comet_scores,
            "OUR_METRIC": self.our_scores
        }

        for name, scores in metrics.items():
            mae, mse, avg = self.compute_similarity(scores)

            results.append({
                "metric": name,
                "mae": round(mae, 2),
                "mse": round(mse, 2),
                "average": round(avg, 2)
            })

        df = pd.DataFrame(results)
        df.to_csv(output_file, sep="\t", index=False)
        print(f"Saved similarity results to {output_file}")

        files.download(output_file)

In [12]:
sim = Similarity(
    bleu_file=bleu_file,
    comet_file=comet_file,
    our_metric_file=our_metric_file,
    human_file=human_file
)

sim.save_results("similarity_results.tsv")

Saved similarity results to similarity_results.tsv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>